### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [26]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("What is agentic AI?")
response

AIMessage(content='**Agentic AI** – short for *agent‑based AI* – is a class of artificial intelligence systems that are designed to act autonomously in an environment, sense information, decide what to do next, and pursue goals or objectives that they (or their designers) have set for them.  \n\nBelow is a quick‑reference guide to what makes an AI “agentic” and why it matters.\n\n| Feature | What it means | Typical examples |\n|---------|---------------|------------------|\n| **Autonomy** | The AI can make decisions without human intervention in real time. | Self‑driving cars, autonomous drones, robotic vacuum cleaners. |\n| **Goal‑orientation** | The system has a clear objective or set of objectives to optimize. | A trading bot that maximizes profit, a game‑playing agent that wins matches. |\n| **Perception** | It senses its environment via sensors or data streams. | Cameras, LiDAR, microphone arrays, API feeds. |\n| **Learning / Adaptation** | It can improve its policy or behavior ov

In [27]:
from langchain.tools import tool
import json

@tool
def get_exchange_rate(currency_pair:str)->str:
    """Get the exchange rate for a currency pair, e.g. 'USD_EUR'"""
    rates = {
        "USD_EUR": "0.873",
        "USD_GBP": "0.758", # British Pound Sterling
        "EUR_GBP": "0.867",
    }
    return rates.get(currency_pair, f"No exchange rate available for {currency_pair}")


model_with_tools=model.bind_tools([get_exchange_rate]) #bind_tools only tells the model that the tool exists (its name, description, and argument schema).

In [28]:
response = model_with_tools.invoke("What's the exchange rate for USD to EUR?")

#### Notice the model hasn't answered yet. 
It only decided to call the tool, and bind_tools never runs the tool for you.

In [29]:
# print(response.pretty_print())
# print(json.dumps(response.model_dump(),indent=2, default=str))
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_exchange_rate
Args: {'currency_pair': 'USD_EUR'}


### Tool Execution Loops

In [30]:
messages = [{"role": "user", "content": "What's the exchange rate for USD to EUR?"}]
ai_msg = model_with_tools.invoke(messages)

In [31]:
ai_msg.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'currency_pair': 'USD_EUR'},
  'id': 'fc_54380b60-497a-4cc0-ba02-5d2014a35dc7',
  'type': 'tool_call'}]

In [32]:
# count=0
# for i in ai_msg.tool_calls:
#     count=count+1
#     print(i,count)

@tool turns your function into a LangChain Tool object, which has its own .invoke() method. It's unrelated to the model's .invoke().

In [33]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the exchange rate for USD to EUR?"}]
ai_msg = model_with_tools.invoke(messages) # → model decides WHICH tool + args
messages.append(ai_msg)
print("Message list in step1 -> ",messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_exchange_rate.invoke(tool_call) # → your code actually RUNS the function
    messages.append(tool_result)

# print("Message list in step2 -> ",json.dumps(messages[0]))
print("Message list in step2 -> ",messages)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print("step3 -->",final_response.text)
# "The exchange rate for USD to EUR is 0.92."

Message list in step1 ->  [{'role': 'user', 'content': "What's the exchange rate for USD to EUR?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_exchange_rate with currency_pair "USD_EUR".', 'tool_calls': [{'id': 'fc_f75efd9c-c3fd-4add-b60a-12a54476369d', 'function': {'arguments': '{"currency_pair":"USD_EUR"}', 'name': 'get_exchange_rate'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 141, 'total_tokens': 184, 'completion_time': 0.045020295, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006902215, 'prompt_tokens_details': None, 'queue_time': 0.315884672, 'total_time': 0.05192251}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a22128-1b8f-72e5-9061-c5f0747856a5-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'currency

In [34]:
messages

[{'role': 'user', 'content': "What's the exchange rate for USD to EUR?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_exchange_rate with currency_pair "USD_EUR".', 'tool_calls': [{'id': 'fc_f75efd9c-c3fd-4add-b60a-12a54476369d', 'function': {'arguments': '{"currency_pair":"USD_EUR"}', 'name': 'get_exchange_rate'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 141, 'total_tokens': 184, 'completion_time': 0.045020295, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006902215, 'prompt_tokens_details': None, 'queue_time': 0.315884672, 'total_time': 0.05192251}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a22128-1b8f-72e5-9061-c5f0747856a5-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'currency_pair': 'USD_EUR'}, 'id':

### 🤔 Before you move on
Print the `messages` list from the cell above — it's not just plain strings, it's a mix of typed objects (a human turn, an AI turn carrying tool_calls, a tool result). What exactly are these message types, and how is a `SystemMessage` different from a `HumanMessage`, `AIMessage`, or `ToolMessage`? That's the whole subject of `4-messages.ipynb`.